In [63]:
import os
import math
import sqlite3
from   ucimlrepo import fetch_ucirepo
from itertools import chain, combinations

def powerset(iterable):
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

path = 'wine.db'
conn = sqlite3.connect('wine.db')


In [64]:
check_table = '''SELECT EXISTS (SELECT name FROM sqlite_schema WHERE  type='table' AND  name='wine_quality');'''

conn = sqlite3.connect('wine.db')
cursor = conn.cursor()
if cursor.execute(check_table).fetchone()[0] == 0:
    wine_quality = fetch_ucirepo(id=186)['data']['original']
    wine_quality.insert(0, "id", range(1, len(wine_quality) + 1))
    wine_quality.to_sql('wine_quality', conn, if_exists='replace', index=True)


In [75]:
player_names = ['fixed_acidity', 'volatile_acidity', 'citric_acid','residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']

path, dist = dict(), dict()

start_id, end_id = 2473,2476

for player_name in player_names:

    get_var   = f'''select avg(power(({player_name} - (select avg({player_name}) from wine_quality)),2)) from wine_quality'''
    get_value = f'''select {player_name} from wine_quality where id = :unique_id'''

    var       = cursor.execute(get_var).fetchone()[0]
    start_val = cursor.execute(get_value, {'unique_id': start_id}).fetchone()[0]
    end_val   =  cursor.execute(get_value, {'unique_id': end_id}).fetchone()[0]

    path[player_name] = [start_val, end_val]
    dist[player_name+'_stddev'] = math.sqrt(var)

    print(player_name.rjust(20),
          str(start_val).ljust(6),
          str(end_val).ljust(6),
          str(round((end_val- start_val)/std_dev,6)).ljust(11),
          round( math.sqrt(var),12))

       fixed_acidity 10.3   6.9    -4.459266   1.296333982238
    volatile_acidity 0.17   0.36   0.249194    0.164623803405
         citric_acid 0.47   0.34   -0.170501   0.145306681008
      residual_sugar 1.4    4.2    3.672336    4.75743757516
           chlorides 0.037  0.018  -0.024919   0.035030905132
 free_sulfur_dioxide 5.0    57.0   68.200532   17.748033750546
total_sulfur_dioxide 33.0   119.0  112.793187  56.517504512656
             density 0.9939 0.9898 -0.005377   0.002998442221
                  pH 2.89   3.28   0.511504    0.16077482767
           sulphates 0.28   0.36   0.104924    0.148794421283
             alcohol 9.6    12.7   4.065801    1.192619955915
             quality 3      9      7.869292    0.873188064445


In [76]:
def value(coalition):

    if len(coalition)==0:
        target = {player_name: path[player_name][0]  for player_name in player_names }
    else:
        target = {player_name: path[player_name][player_name in coalition]  for player_name in player_names }

    params = target | dist

    query = f'''select quality
                from wine_quality
                order by    power((fixed_acidity - :fixed_acidity)/:fixed_acidity_stddev ,2)
                          + power((volatile_acidity - :volatile_acidity)/:volatile_acidity_stddev ,2)
                          + power((citric_acid - :citric_acid)/:citric_acid_stddev ,2)
                          + power((residual_sugar - :residual_sugar)/:residual_sugar_stddev ,2)
                          + power((chlorides - :chlorides)/:chlorides_stddev ,2)
                          + power((free_sulfur_dioxide - :free_sulfur_dioxide)/:free_sulfur_dioxide_stddev ,2)
                          + power((total_sulfur_dioxide - :total_sulfur_dioxide)/:total_sulfur_dioxide ,2)
                          + power((density - :density)/ :density_stddev ,2)
                          + power((pH - :pH)/:pH_stddev,2)
                          + power((sulphates - :sulphates)/:sulphates_stddev,2)
                          + power((alcohol - :alcohol)/:alcohol_stddev,2) '''



    return cursor.execute(query, params).fetchone()[0]




In [77]:
def gamma(players, coalition):
    N = len(players)
    S = len(coalition)
    return math.factorial(S) * math.factorial(N - S - 1) / math.factorial(N)

player_set = set(player_names) - {'quality'}
phi = 0
for player in player_set:
    coalitions = player_set - {player}
    phi_i=0.0
    for S in powerset(coalitions):
        v2 = value(set(S).union({player}))
        v1 = value(set(S))
        phi_i += gamma(player_set, S) * (v2 - v1)

    print(player, phi_i)
    phi += phi_i

print(phi)


chlorides 0.07059884559884562
fixed_acidity 0.8356782106782108
free_sulfur_dioxide 1.9289321789321792
sulphates 0.009884559884559871
citric_acid 0.08726551226551227
pH 1.0467893217893223
alcohol 1.0852813852813847
volatile_acidity 0.16663059163059166
density 0.22536075036075054
residual_sugar 0.0047258297258297305
total_sulfur_dioxide 0.5388528138528141
6.0


In [67]:
conn.close()

In [70]:
wine_quality = fetch_ucirepo(id=186)['data']['original']
wine_quality.insert(0, "id", range(1, len(wine_quality) + 1))